# Stock Cluster Analysis

Identifies peer groups across the US equity universe (>\$100M market cap) by clustering
stocks on four complementary feature families:

| Family | Features |
|--------|----------|
| **Return behavior** | Annualized vol, Sharpe, max drawdown, skew/kurt |
| **Momentum** | 4w / 13w / 26w / 52w / Jegadeesh-Titman |
| **Factor loadings** | Market beta + PCA latent factors (alpha, R²) |
| **Fundamentals** | Size (log mkt cap), P/E, revenue growth, book/price |

The temporal engine re-clusters every 4 weeks on a 52-week rolling window, so you can
answer *"who were AAPL's peers in Jan 2023 vs Jan 2025, and how has that changed?"*

**Prerequisites:** A live Bloomberg Terminal connection with `xbbg` installed.
```
pip install -r requirements.txt
```

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import warnings
import logging

import numpy as np
import pandas as pd
import plotly.io as pio

from IPython.display import display

from src import bloomberg_data as bbg
from src import features as feat
from src import clustering as clust
from src import viz

pio.renderers.default = 'notebook'
logging.basicConfig(level=logging.INFO, format='%(levelname)s  %(message)s')
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.4f}'.format)

print('Environment ready.')

## Step 0: Verify Bloomberg Connection

**Run this before anything else.** You should see `Bloomberg connected` printed below.

In [ ]:
# ── BLOOMBERG CONNECTION TEST ─────────────────────────────────────────────────
# Run this cell first. It fetches one field for SPY to confirm Terminal is live.
# If it fails, make sure Bloomberg Terminal is open and you are logged in.

connected = bbg.test_connection()
if not connected:
    raise SystemExit('Fix Bloomberg connection before continuing.')

## 1. Configuration

Adjust these parameters before running the rest of the notebook.

In [ ]:
# ── Bloomberg universe ────────────────────────────────────────────────────────
INDEX         = 'RAY Index'   # Russell 3000 (covers ~97% of US equity mkt cap)
MIN_MKTCAP_MM = 100           # minimum market cap ($M)

# ── Date range (4 years back) ─────────────────────────────────────────────────
START_DATE  = '20220501'
END_DATE    = '20260520'
PERIODICITY = 'WEEKLY'

# ── Clustering parameters ─────────────────────────────────────────────────────
WINDOW_WEEKS  = 52    # rolling feature / clustering window
STEP_WEEKS    = 4     # re-cluster every N weeks
N_CLUSTERS    = 25    # initial k (refined by silhouette analysis below)
N_PCA_DIMS    = 30    # PCA reduction before KMeans
N_PCA_FACTORS = 3     # latent PCA factors in the factor-loading model

# ── Peer finder ───────────────────────────────────────────────────────────────
N_PEERS = 20

print('Config loaded.')

## 2. Data Collection

All Bloomberg calls are cached to `data/raw/` as Parquet files.
Re-running any cell will use the cache unless `use_cache=False` is passed.

In [ ]:
# Download (or load cached) Russell 3000 filtered to >= $100M market cap
tickers = bbg.get_universe(
    index=INDEX,
    min_market_cap_mm=MIN_MKTCAP_MM,
)
print(f'Universe: {len(tickers):,} tickers')
print('Sample:', tickers[:5])

In [ ]:
# Weekly adjusted closing prices — batches of 200 tickers to respect Bloomberg limits
prices = bbg.get_price_history(
    tickers,
    start_date=START_DATE,
    end_date=END_DATE,
    periodicity=PERIODICITY,
)
print(f'Prices: {prices.shape[0]} weeks x {prices.shape[1]:,} tickers')
print(f'Date range: {prices.index[0].date()} -> {prices.index[-1].date()}')
prices.tail(3)

In [ ]:
# Current-period fundamentals (market cap, P/E, growth, sector, etc.)
fundamentals = bbg.get_fundamentals(tickers)
print(f'Fundamentals: {fundamentals.shape[0]:,} tickers x {fundamentals.shape[1]} fields')
fundamentals.head(3)

In [ ]:
# Market factor proxy (SPY) for beta/alpha computation
market_returns = bbg.get_market_factor(start_date=START_DATE, end_date=END_DATE)
print(f'Market factor: {len(market_returns)} weekly observations')
print(f'Annualized vol: {market_returns.std() * (52 ** 0.5):.1%}')

In [ ]:
returns = feat.compute_returns(prices)

print('=== Data Summary ===')
print(f'Return matrix : {returns.shape[0]} x {returns.shape[1]:,}')
print(f'Avg weekly ret: {returns.stack().mean():.3%}')
print(f'Avg weekly vol: {returns.stack().std():.3%}')

if 'gics_sector_name' in fundamentals.columns:
    print('\nSector breakdown:')
    display(fundamentals['gics_sector_name'].value_counts())

## 3. Feature Engineering

For the **current snapshot** we build features on the most recent `WINDOW_WEEKS` of data.
The temporal engine in Section 6 will repeat this at every step.

In [ ]:
feature_matrix = feat.build_feature_matrix(
    returns=returns,
    market_returns=market_returns,
    fundamentals=fundamentals,
    window=WINDOW_WEEKS,
    n_pca_factors=N_PCA_FACTORS,
)
norm_features = feat.normalize_features(feature_matrix)

print(f'Feature matrix : {feature_matrix.shape[0]:,} stocks x {feature_matrix.shape[1]} features')
print(f'Features: {list(feature_matrix.columns)}')

In [ ]:
feature_matrix.describe().round(4)

## 4. Selecting the Number of Clusters

We evaluate K-Means silhouette scores over a range of k values.  
The silhouette score measures how well each stock fits its own cluster vs. neighboring clusters (+1 = perfect, -1 = wrong cluster).

In [ ]:
# PCA-reduce features before clustering (improves clustering quality + speed)
X_pca, pca_model = clust.pca_reduce(norm_features, n_components=N_PCA_DIMS)

explained = pca_model.explained_variance_ratio_.cumsum()
print(f'PCA: {N_PCA_DIMS} components explain {explained[-1]:.1%} of total variance')

In [ ]:
# Evaluate k from 5 to 50 in steps of 5 (takes ~2-3 minutes for a large universe)
k_scores = clust.score_k_range(X_pca, k_range=range(5, 51, 5))
viz.plot_k_selection(k_scores).show()

OPTIMAL_K = clust.best_k(k_scores)
print(f'\nBest k by silhouette: {OPTIMAL_K}')
print('Override OPTIMAL_K below if you prefer a different granularity.')

In [ ]:
# Optional manual override
# OPTIMAL_K = 30

print(f'Using k = {OPTIMAL_K}')

## 5. Static Cluster Analysis (Most Recent Period)

Cluster the universe using the latest `WINDOW_WEEKS` of data.

In [ ]:
labels_arr = clust.cluster(X_pca, n_clusters=OPTIMAL_K, method='kmeans')
cluster_labels = pd.Series(labels_arr, index=norm_features.index, name='cluster')

print(f'Cluster distribution (k={OPTIMAL_K}):')
display(cluster_labels.value_counts().sort_index().to_frame('count'))

In [ ]:
# 2D projection for visualization (UMAP if available, else PCA)
try:
    import umap as umap_lib
    reducer = umap_lib.UMAP(
        n_components=2, n_neighbors=20, min_dist=0.05,
        metric='euclidean', random_state=42
    )
    coords_2d = reducer.fit_transform(X_pca)
    projection_method = 'UMAP'
except ImportError:
    from sklearn.decomposition import PCA as _PCA2D
    coords_2d = _PCA2D(n_components=2, random_state=42).fit_transform(X_pca)
    projection_method = 'PCA'

print(f'Projection: {projection_method}')

viz.plot_cluster_map(
    coords_2d, cluster_labels, fundamentals,
    title=f'Stock Clusters — {projection_method} projection  (k={OPTIMAL_K}, window={WINDOW_WEEKS}w)'
).show()

In [ ]:
viz.plot_cluster_profiles(feature_matrix, cluster_labels).show()

In [ ]:
# Summary table: one row per cluster
rows = []
for cid in sorted(cluster_labels.unique()):
    members = cluster_labels[cluster_labels == cid].index.tolist()
    avg = feature_matrix.loc[members].mean()
    row = {
        'cluster': cid,
        'n_stocks': len(members),
        'avg_ann_vol': avg.get('ann_vol', float('nan')),
        'avg_sharpe': avg.get('sharpe', float('nan')),
        'avg_mom_13w': avg.get('mom_13w', float('nan')),
        'avg_beta': avg.get('beta_market', float('nan')),
        'avg_log_mktcap': avg.get('log_mkt_cap', float('nan')),
    }
    if 'gics_sector_name' in fundamentals.columns:
        top_sector = (
            fundamentals.loc[fundamentals.index.isin(members), 'gics_sector_name']
            .value_counts().idxmax()
        )
        row['top_sector'] = top_sector
    rows.append(row)

cluster_table = pd.DataFrame(rows).set_index('cluster').round(3)
display(cluster_table)

## 6. Peer Finder

Given a ticker and snapshot date, returns its cluster peers ranked by return correlation.

Set `TARGET_TICKER` to any Bloomberg equity ticker in the universe.

In [ ]:
TARGET_TICKER = 'AAPL US Equity'   # ← change to any ticker

snap_date = returns.index[-1]

# Wrap the current snapshot into the assignment format expected by find_peers
snapshot_df = pd.DataFrame({snap_date: cluster_labels}).T

peers_df = clust.find_peers(
    ticker=TARGET_TICKER,
    cluster_assignments=snapshot_df,
    returns=returns,
    as_of_date=snap_date,
    n_peers=N_PEERS,
)

print(f"Top {N_PEERS} peers of {TARGET_TICKER} as of {snap_date.date()}")
print(f"Cluster {peers_df['cluster'].iloc[0]} — {len(peers_df)} peers shown")
display(peers_df)

In [ ]:
# Show target + top 10 peers on a normalized performance chart
viz.plot_peer_performance(
    ticker=TARGET_TICKER,
    peers=peers_df['ticker'].tolist()[:10],
    returns=returns,
    weeks=52,
).show()

In [ ]:
# Highlight the target ticker on the cluster map
viz.plot_cluster_map(
    coords_2d, cluster_labels, fundamentals,
    highlight_tickers=[TARGET_TICKER],
    title=f'{TARGET_TICKER} in the Cluster Map'
).show()

## 7. Temporal Analysis — How Clusters Evolve Over Time

Re-run K-Means every `STEP_WEEKS` using a trailing `WINDOW_WEEKS` window.
Hungarian matching keeps cluster IDs consistent across periods.

> **Note:** For ~2,500 stocks over 4 years this typically takes 10-30 minutes.
> Run once and reload from the cached Parquet file on subsequent sessions.

In [ ]:
from pathlib import Path

ASSIGNMENTS_PATH = Path('data/processed/cluster_assignments.parquet')
ASSIGNMENTS_PATH.parent.mkdir(parents=True, exist_ok=True)

if ASSIGNMENTS_PATH.exists():
    cluster_assignments = pd.read_parquet(ASSIGNMENTS_PATH)
    print(f'Loaded cached assignments: {cluster_assignments.shape[0]} snapshots x {cluster_assignments.shape[1]:,} stocks')
    print(f'Date range: {cluster_assignments.index[0].date()} -> {cluster_assignments.index[-1].date()}')
else:
    print('No cache found — run the next cell to compute rolling assignments.')
    print(f'Expected time: 15-30 min for {len(tickers):,} tickers')

In [ ]:
# Skip this cell if the previous cell loaded from cache.
# Uncomment to re-run even if cache exists.

if not ASSIGNMENTS_PATH.exists():
    print(f'Running rolling cluster analysis ({(len(returns) - WINDOW_WEEKS) // STEP_WEEKS} snapshots)...')
    cluster_assignments = clust.run_rolling_clustering(
        returns=returns,
        market_returns=market_returns,
        fundamentals=fundamentals,
        window_weeks=WINDOW_WEEKS,
        step_weeks=STEP_WEEKS,
        n_clusters=N_CLUSTERS,
        n_pca_dims=N_PCA_DIMS,
        n_pca_factors=N_PCA_FACTORS,
    )
    cluster_assignments.to_parquet(ASSIGNMENTS_PATH)
    print(f'Done. Saved to {ASSIGNMENTS_PATH}')
    print(f'Shape: {cluster_assignments.shape}')

In [ ]:
# Track a single stock across all time periods
TRACK_TICKER = TARGET_TICKER  # same as peer finder, or set a different one

peer_hist = clust.get_peer_history(TRACK_TICKER, cluster_assignments)

print(f'Cluster history for {TRACK_TICKER}:')
display(peer_hist.tail(12).round(3))

viz.plot_cluster_history(TRACK_TICKER, peer_hist).show()

In [ ]:
# Heatmap of cluster membership over time for the target + its current peers
heatmap_tickers = [TRACK_TICKER] + peers_df['ticker'].tolist()[:20]
heatmap_tickers = [t for t in heatmap_tickers if t in cluster_assignments.columns]

viz.plot_cluster_membership_heatmap(
    cluster_assignments,
    tickers=heatmap_tickers,
).show()

### Time-Travel Peer Finder

Compare who a stock clustered with at different points in history.

In [ ]:
comparison_dates = [
    pd.Timestamp('2023-01-01'),
    pd.Timestamp('2024-01-01'),
    pd.Timestamp('2025-01-01'),
    cluster_assignments.index[-1],  # most recent snapshot
]

print(f'Peer comparison for {TRACK_TICKER}\n')
print(f'{"Date":<14} {"Cluster":>8}  Top 8 Peers')
print('-' * 80)

for dt in comparison_dates:
    try:
        p = clust.find_peers(
            ticker=TRACK_TICKER,
            cluster_assignments=cluster_assignments,
            returns=returns,
            as_of_date=dt,
            n_peers=8,
        )
        peer_list = ', '.join(p['ticker'].tolist())
        print(f'{str(dt.date()):<14} {p["cluster"].iloc[0]:>8}  {peer_list}')
    except Exception as e:
        print(f'{str(dt.date()):<14}   skipped: {e}')

In [ ]:
# Performance of the current peer group over different look-back windows
peer_tickers = peers_df['ticker'].tolist()[:8]

for weeks, label in [(13, '3 months'), (26, '6 months'), (52, '1 year')]:
    fig = viz.plot_peer_performance(
        ticker=TRACK_TICKER,
        peers=peer_tickers,
        returns=returns,
        weeks=weeks,
        title=f'{TRACK_TICKER} vs Peers — {label}',
    )
    fig.show()

## 8. Cluster Deep-Dive

Inspect any cluster by ID to see its full membership, sector breakdown, and average characteristics.

In [ ]:
# Set INSPECT_CLUSTER to any cluster ID from the table above
INSPECT_CLUSTER = int(cluster_labels[TARGET_TICKER])

composition = clust.cluster_composition(
    cluster_id=INSPECT_CLUSTER,
    cluster_assignments=snapshot_df,
    fundamentals=fundamentals,
)
print(f'Cluster {INSPECT_CLUSTER}: {len(composition)} stocks')
if 'gics_sector_name' in composition.columns:
    print('\nSector breakdown:')
    display(composition['gics_sector_name'].value_counts())

display(composition.head(30))

In [ ]:
# Average feature values for the selected cluster vs the universe average
cluster_members = cluster_labels[cluster_labels == INSPECT_CLUSTER].index
cluster_avg = feature_matrix.loc[cluster_members].mean().rename('cluster_avg')
universe_avg = feature_matrix.mean().rename('universe_avg')

comparison = pd.concat([cluster_avg, universe_avg], axis=1)
comparison['vs_universe'] = comparison['cluster_avg'] - comparison['universe_avg']
display(comparison.round(4))

## 9. Export Results

In [ ]:
from pathlib import Path

out_dir = Path('data/processed')
out_dir.mkdir(parents=True, exist_ok=True)

# Current snapshot: ticker -> cluster
cluster_labels.to_frame('cluster').to_parquet(out_dir / 'current_clusters.parquet')

# Full feature matrix
feature_matrix.to_parquet(out_dir / 'feature_matrix.parquet')

# Cluster summary table
cluster_table.to_parquet(out_dir / 'cluster_summary.parquet')

print('Saved:')
for f in out_dir.iterdir():
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:<45} {size_kb:>8.1f} KB')

## Notes & Next Steps

### What this tells you
- **Current peers**: stocks that share return characteristics, factor loadings, momentum regime, and size/valuation profile with your target.
- **Peer churn**: high churn weeks indicate regime changes — the stock is re-categorizing itself in the market's mind.
- **Cluster drift**: a stock migrating from a low-vol / value cluster to a high-beta / growth cluster often precedes multiple expansion or compression.

### Limitations
- **Survivorship bias**: using the current Russell 3000 excludes stocks that were delisted. For longer backtests, use a point-in-time index from Bloomberg.
- **Quarterly fundamentals**: P/E and growth figures are current-period; a future enhancement is to pull quarterly BDH data for point-in-time fundamentals.
- **k sensitivity**: results are moderately sensitive to k. The silhouette score is a good guide but consider also testing k values that give economically intuitive cluster sizes (15-30 stocks per cluster is often most useful).

### Suggested enhancements
1. **Sector-neutral clustering** — run separate clustering within each GICS sector to find intra-sector outliers.
2. **Graph-based clustering** — build a correlation network (edges = correlation > 0.6) and use community detection (Louvain algorithm) as an alternative to K-Means.
3. **Cluster transition signals** — backtest whether cluster-switching events have alpha.
4. **Factor tilt analysis** — for each cluster, compute the aggregate factor tilt vs. the benchmark to identify implicit factor bets.